# SCF Force Consistency

This notebook checks whether the force reported by the toy DFT engine is consistent with finite differences of the SCF total energy. The check is deliberately small and diagnostic: it validates the current Gaussian-center model, not production DFT force accuracy.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.dft import DFTSystem, DiracExchange, SCFConfig, run_scf, scf_total_energy_forces

The helper displaces each Gaussian center by `±δ`, reruns SCF, and compares `-[E(R+δ)-E(R-δ)]/2δ` against the force returned by `run_scf(...)`.

In [ ]:
system = DFTSystem.one_center(
    cell=(6.0, 6.0, 6.0),
    grid_shape=(4, 4, 4),
    center=(3.0, 3.0, 3.0),
    amplitude=-2.0,
    width=0.8,
)
config = SCFConfig(max_iterations=2, solver="dense", seed=7)
check = scf_total_energy_forces(
    system,
    config=config,
    xc_functional=DiracExchange(),
    displacement=1e-3,
)
{
    "max_abs_error": check.max_abs_error,
    "rms_abs_error": check.rms_abs_error,
    "analytic_forces": np.array(check.analytic_forces).tolist(),
    "finite_difference_forces": np.array(check.finite_difference_forces).tolist(),
}

A one-center system should have a symmetric energy curve around the cell center. The plotted curve is a sanity check on the total-energy surface used by the finite-difference force.

In [ ]:
xs = np.linspace(2.8, 3.2, 7)
energies = []
for x in xs:
    shifted = system.with_centers([[x, 3.0, 3.0]])
    energies.append(run_scf(shifted, config=config, xc_functional=DiracExchange()).total_energy)

fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
ax.plot(xs, energies, marker="o")
ax.axvline(3.0, color="black", linewidth=1, linestyle="--")
ax.set_xlabel("center x")
ax.set_ylabel("SCF total energy")
ax.set_title("one-center energy surface");